In [1]:
# %load_ext autoreload
# %autoreload 2

In [2]:
!pip install -q torch transformers sentence-transformers pypdf2 faiss-cpu langchain tqdm
print("Installed required packages.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 84.3 MB/s eta 0:00:00
Installed required packages.


In [3]:
# 라이브러리 및 헬퍼 모듈 로딩 셀
# - 한글 폰트 설정 및 Google Drive 연결(Colab 환경)
# - helper_utils.py, helper_c0z0c_dev.py 최신 버전 다운로드 및 import
# - 항상 importlib.reload로 헬퍼 모듈을 새로 읽어서 사용
# - from helper_utils import * / from helper_c0z0c_dev import *로 함수 직접 사용
# - 코드/경로/저장 로직은 헬퍼 함수(get_path_modeling 등)로 통일

import importlib
from urllib.request import urlretrieve

urlretrieve("https://raw.githubusercontent.com/c0z0c/jupyter_hangul/refs/heads/beta/helper_utils.py", "helper_utils.py")
import helper_utils as hu
from helper_utils import *

urlretrieve("https://raw.githubusercontent.com/c0z0c/jupyter_hangul/refs/heads/beta/helper_c0z0c_dev.py", "helper_c0z0c_dev.py")
import importlib
import helper_c0z0c_dev as helper

helper = importlib.reload(helper)
from helper_c0z0c_dev import *

hu = importlib.reload(hu)
from helper_utils import *


helper_utils.py loaded
🌐 https://c0z0c.github.io/jupyter_hangul
ℹ️ NumPy 2.0.2 (v2.x+): 호환성 모드 적용됨
install fonts-nanum...
Mounted at /content/drive
✅ 설정 완료: 한글 폰트, plt 전역 등록, pandas 확장, 캐시 기능
pd commit 저장 경로 = /content/drive/MyDrive
🌐 https://c0z0c.github.io/jupyter_hangul
ℹ️ NumPy 2.0.2 (v2.x+): 호환성 모드 적용됨
Mounted at /content/drive
✅ 설정 완료: 한글 폰트, plt 전역 등록, pandas 확장, 캐시 기능
pd commit 저장 경로 = /content/drive/MyDrive
helper_utils.py loaded


In [4]:
# 코랩 휴지통 비우기
# 1. Google Drive 인증
def clear_google_drive_trash():
    if IS_COLAB:
        # print("코랩 환경에서 Google Drive 휴지통 비우기를 시작합니다.")
        from google.colab import auth
        auth.authenticate_user()

        # 2. Google Drive API v3 서비스 빌드
        # 'drive', 'v3'은 Drive API와 버전을 지정합니다.
        from googleapiclient.discovery import build
        drive_service = build('drive', 'v3')

        # 3. 휴지통 비우기 명령어 실행
        # files().emptyTrash().execute() 메서드를 호출하여 휴지통을 영구 삭제합니다.
        drive_service.files().emptyTrash().execute()
        # print("Google Drive 휴지통을 성공적으로 비웠습니다.")

clear_google_drive_trash()

In [5]:

# 기본 라이브러리

# --- Scikit-learn: 데이터 전처리, 모델, 평가 ---
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.datasets import (
    fetch_california_housing, load_iris, make_moons, make_circles,
    load_breast_cancer, load_wine
)
from sklearn import datasets
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.metrics import average_precision_score

# --- 기타 라이브러리 ---
from PIL import Image
from PIL import ImageFilter
from PIL import ImageDraw
import albumentations as A
import IPython.display
#from tqdm import tqdm
from tqdm.notebook import tqdm

# --- PyTorch: 딥러닝 관련 ---
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import argparse

from datasets import Dataset
from torch.utils.data import DataLoader
from torchvision.transforms import v2
from torchvision.datasets import CocoDetection
from torchvision.transforms import functional as TF
from torch.nn import CrossEntropyLoss
from collections import OrderedDict
from datasets import DatasetDict
from torch.utils.data import DataLoader
from torch.optim import AdamW
from datasets import load_dataset
from transformers import AutoTokenizer

from sklearn.utils.class_weight import compute_class_weight

# --- 기타 ---
import re
import os
import copy
import sys
import json
import math
import random
import yaml
import shutil
import pandas as pd
import numpy as np
import zipfile
from typing import Union, List, Optional, Tuple
from pathlib import Path

import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
import seaborn as sns
from datetime import datetime
from datetime import timezone, timedelta
import pytz

from dataclasses import dataclass, asdict
from transformers import (
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import (
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)
import wandb

_kst = pytz.timezone('Asia/Seoul')

# GPU 설정
_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_device_cpu = torch.device('cpu')

  # 재현 가능한 결과를 위해
np.random.seed(42)
torch.manual_seed(42)
if _device == 'cuda':
    torch.cuda.manual_seed_all(42)

logger = logging.getLogger(__name__)
#logger = logging.getLogger()
#logger.setLevel(logging.DEBUG)
if IS_COLAB:
  logger.setLevel(logging.INFO)
else:
  logger.setLevel(logging.DEBUG)
logger.info(f"라이브러리 로드 완료 사용장치:{_device}")


2025-11-03 16:00:47 [I] 라이브러리 로드 완료 사용장치:cuda


In [6]:
# WandB 환경변수 설정 및 설치 셀
# 목적: wandb 사용 시 코드/모델/메타데이터 자동 업로드 및 불필요한 로그를 차단하여 보안과 재현성 강화
# 주요 환경변수 설명:
#   - WANDB_DISABLE_CODE: 코드 자동 저장 차단 (보안)
#   - WANDB_SILENT: wandb 출력 최소화 (로그 정리)
#   - WANDB_IGNORE_GLOBS: 모델 파일 자동 업로드 차단 (*.pth, *.pt 등)
#   - WANDB_DISABLE_SERVICE: 패키지/메타데이터 수집 비활성화
#   - WANDB_DISABLE_PIP: pip 정보 수집 차단
#   - WANDB_DISABLE_STATS, WANDB_DISABLE_META: wandb 내부 통계/메타데이터 비활성화

import os
os.environ['WANDB_DISABLE_CODE'] = 'true'
os.environ['WANDB_SILENT'] = 'true'
os.environ['WANDB_IGNORE_GLOBS'] = '*.pth,*.pt,*.ckpt,*.bin'
os.environ['WANDB_DISABLE_SERVICE'] = 'true'
os.environ['WANDB_DISABLE_PIP'] = 'true'
os.environ['WANDB_DISABLE_STATS'] = 'true'
os.environ['WANDB_DISABLE_META'] = 'true'

try:
    import wandb
    logger.debug(f"wandb 설치됨: 버전 {wandb.__version__} — 설치 생략")
except Exception:
    logger.debug("wandb 미설치 감지 — 설치 시작...")
    !pip install -q wandb
    logger.debug("설치 완료: wandb")
    import wandb

!pip install -q --upgrade wandb

importlib.reload(wandb)
import wandb

logger.debug(f"WANDB_DISABLE_CODE {os.environ['WANDB_DISABLE_CODE']}")
logger.debug(f"WANDB_SILENT {os.environ['WANDB_SILENT']}")
logger.debug(f"WANDB_IGNORE_GLOBS {os.environ['WANDB_IGNORE_GLOBS']}")
logger.debug(f"WANDB_DISABLE_SERVICE {os.environ['WANDB_DISABLE_SERVICE']}")
logger.debug(f"WANDB_DISABLE_PIP {os.environ['WANDB_DISABLE_PIP']}")
logger.debug(f"WANDB_DISABLE_STATS {os.environ['WANDB_DISABLE_STATS']}")
logger.debug(f"WANDB_DISABLE_META {os.environ['WANDB_DISABLE_META']}")

logger.info("업그레이드 완료: wandb")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 127.1 MB/s eta 0:00:00


2025-11-03 16:00:57 [I] 업그레이드 완료: wandb


In [7]:
# [WandB 로그인 셀]
# Colab 환경에서는 Colab 비밀(환경변수)에서 WANDB_API_KEY를 읽어 wandb 로그인에 사용합니다.
# 로컬 환경에서는 .env 파일에서 읽어 환경변수로 설정합니다.
# API 키는 직접 노출하지 않고, 환경변수로만 처리합니다.
logger.setLevel(logging.DEBUG)

api_key = None
if IS_COLAB:
    from google.colab import userdata
    api_key = userdata.get('WANDB_API_KEY')
else:
    from dotenv import load_dotenv
    load_dotenv()  # .env 파일 자동 로드
    api_key = os.getenv("WANDB_API_KEY")

if api_key:
    api_key = api_key.strip()
    os.environ["WANDB_API_KEY"] = api_key
    logger.debug(f"WANDB_API_KEY [{api_key[:4]}****{api_key[-4:]}] 환경변수 설정 완료")

    # 수정: Python API로 로그인
    import wandb
    wandb.login(key=api_key, relogin=True)
    logger.info("로그인 완료: wandb")
else:
    logger.warning("WANDB_API_KEY가 설정되지 않아 wandb 로그인 생략됨")

2025-11-03 16:01:01 [D] WANDB_API_KEY [86a7****cb5d] 환경변수 설정 완료
2025-11-03 16:01:04 [I] 로그인 완료: wandb


In [8]:
import PyPDF2
from typing import List

def load_pdf(pdf_path: str) -> str:
    """
    PDF 파일에서 텍스트를 추출합니다.

    Args:
        pdf_path: PDF 파일 경로

    Returns:
        추출된 전체 텍스트
    """
    text = ""

    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)

            # 모든 페이지의 텍스트 추출
            for page_num, page in enumerate(pdf_reader.pages):
                page_text = page.extract_text()
                text += f"\n--- 페이지 {page_num + 1} ---\n"
                text += page_text

        print(f"PDF 로딩 완료: {len(pdf_reader.pages)}페이지, {len(text)}자")
        return text

    except FileNotFoundError:
        print(f"파일을 찾을 수 없습니다: {pdf_path}")
        return ""
    except Exception as e:
        print(f"PDF 로딩 오류: {e}")
        return ""

# 사용 예시
data_pdf_path = str(Path(drive_root()) / 'data' / 'data.pdf')
logger.info(f"PDF 경로: {Path(data_pdf_path).exists()} {data_pdf_path}")

text_pdf = load_pdf(data_pdf_path)
print(text_pdf[:500])  # 처음 500자 출력


2025-11-03 16:01:21 [I] PDF 경로: True /content/drive/MyDrive/data/data.pdf


PDF 로딩 완료: 4페이지, 8965자

--- 페이지 1 ---
영상정보관리사           ◐2023년 04월 02일 필기 기출문제 ◑ 전자문제집 CBT : www.comcbt.com
최강 자격증 기출문제 전자문제집 CBT : www.comcbt.com1. 지방자치단체 CCTV 통합관제센터 관제요원의 업무에 해당하
지 않는 것은?
   ① 특이사항 발생 시 보고 후 사건 현장 출동
   ② 실시간 전송되는 영상자료 모니터링
   ③ 방범, 어린이 보호용 등 CCTV 모니터링
   ④ 기타 담당 공무원의 지시에 따라 근무
2. 한국직업사전에 제시된 카지노감시운영원의 수행직무에 해당
하지 않는 것은?
   ① 녹화 장비를 설치 및 유지보수하고 녹화물을 정리·관리한
다.
   ② 감시카메라를 통해 고객을 감시하고 카지노 종사원의 업
무 및 회사의 자산을 감시하여 위반사항을 관찰, 기록 및 
보고한다 .
   ③ 게임 테이블 , 케이지 , 카운터 룸 등 카지노 영업장 내에 
설치되어 있는 감시카메라 (CCTV) 로 각 영


In [9]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 40.8 MB/s eta 0:00:00


In [10]:
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

def create_sample_korean_pdf(filename: str = "sample_korean.pdf"):
    """한국어 테스트용 PDF 생성"""

    # 한글 폰트 등록 (시스템에 따라 경로 조정)
    # Windows: C:/Windows/Fonts/malgun.ttf
    # Mac: /Library/Fonts/AppleGothic.ttf
    # Linux: /usr/share/fonts/truetype/nanum/NanumGothic.ttf

    c = canvas.Canvas(filename, pagesize=A4)

    # 폰트 설정 시도
    try:
        if IS_COLAB:
            pdfmetrics.registerFont(TTFont('Korean', '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'))
        else:
            pdfmetrics.registerFont(TTFont('Korean', 'NanumGothic.ttf'))
        c.setFont('Korean', 12)
    except:
        print("한글 폰트 없음. 기본 폰트 사용")
        c.setFont('Helvetica', 12)

    # 내용 작성
    y = 800
    content = [
        "인공지능과 RAG 시스템",
        "",
        "RAG(Retrieval-Augmented Generation)는 검색 증강 생성 기술입니다.",
        "이 기술은 대규모 언어 모델의 한계를 극복하기 위해 개발되었습니다.",
        "",
        "주요 특징:",
        "1. 외부 지식 베이스 활용",
        "2. 환각(Hallucination) 감소",
        "3. 최신 정보 제공 가능",
        "",
        "ExaOne은 LG AI연구원이 개발한 한국어 언어 모델입니다.",
        "경량화된 크기로 개인 PC에서도 실행 가능합니다.",
    ]

    for line in content:
        c.drawString(50, y, line)
        y -= 20

    c.save()
    print(f"테스트 PDF 생성: {filename}")
    return filename

# 실행
create_sample_korean_pdf()

테스트 PDF 생성: sample_korean.pdf


'sample_korean.pdf'

In [11]:
!pip install -q --upgrade langchain
!pip install -q langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.3/469.3 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.6/207.6 kB 23.8 MB/s eta 0:00:00


In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_text(text: str, chunk_size: int = 500, chunk_overlap: int = 50) -> List[str]:
    """
    텍스트를 의미 있는 청크로 분할합니다.

    Args:
        text: 분할할 텍스트
        chunk_size: 청크 크기 (문자 수)
        chunk_overlap: 청크 간 겹침 (문맥 유지)

    Returns:
        분할된 텍스트 리스트
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    chunks = text_splitter.split_text(text)

    print(f"텍스트 분할 완료: {len(chunks)}개 청크")
    print(f"   평균 청크 크기: {sum(len(c) for c in chunks) // len(chunks)}자")

    return chunks

text = load_pdf(data_pdf_path)
#text = load_pdf('sample_korean.pdf')
chunks = split_text(text, chunk_size=300, chunk_overlap=30)

# 청크 확인
for i, chunk in enumerate(chunks[:3]):
    print(f"\n[청크 {i+1}]")
    print(chunk)
    print(f"길이: {len(chunk)}자")

PDF 로딩 완료: 4페이지, 8965자
텍스트 분할 완료: 34개 청크
   평균 청크 크기: 272자

[청크 1]
--- 페이지 1 ---
영상정보관리사           ◐2023년 04월 02일 필기 기출문제 ◑ 전자문제집 CBT : www.comcbt.com
최강 자격증 기출문제 전자문제집 CBT : www.comcbt.com1. 지방자치단체 CCTV 통합관제센터 관제요원의 업무에 해당하
지 않는 것은?
   ① 특이사항 발생 시 보고 후 사건 현장 출동
   ② 실시간 전송되는 영상자료 모니터링
   ③ 방범, 어린이 보호용 등 CCTV 모니터링
   ④ 기타 담당 공무원의 지시에 따라 근무
길이: 274자

[청크 2]
④ 기타 담당 공무원의 지시에 따라 근무
2. 한국직업사전에 제시된 카지노감시운영원의 수행직무에 해당
하지 않는 것은?
   ① 녹화 장비를 설치 및 유지보수하고 녹화물을 정리·관리한
다.
   ② 감시카메라를 통해 고객을 감시하고 카지노 종사원의 업
무 및 회사의 자산을 감시하여 위반사항을 관찰, 기록 및 
보고한다 .
   ③ 게임 테이블 , 케이지 , 카운터 룸 등 카지노 영업장 내에 
설치되어 있는 감시카메라 (CCTV) 로 각 영업장을 녹화한
다.
   ④ 필요시 증거물로 녹화물을 제공한다 .
길이: 283자

[청크 3]
다.
   ④ 필요시 증거물로 녹화물을 제공한다 .
3. 다음 중 지방자치단체에서 구축한 CCTV 관제센터의 운영과 
거리가 먼 것은?
   ① 자율방범대 , 녹색어머니연합회 , 초등학교 운영위원회 등 
주민들을 대상으로 관제센터 주민홍보 실시
   ② 관내 어린이집 ·유치원 원아, 초등학생을 대상으로 시청각 
교육과 체험학습을 병행한 어린이 안전체험관 운영
   ③ 정보주체 유무와 상관없이 시민들에게 CCTV 영상정보를 
손쉽게 열람 또는 제공 등의 주민편익 도모
길이: 262자


In [13]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

class VectorStore:
    """FAISS 기반 벡터 저장소"""

    def __init__(self, model_name: str = "paraphrase-multilingual-MiniLM-L12-v2"):
        """
        Args:
            model_name: 임베딩 모델 이름
                - paraphrase-multilingual-MiniLM-L12-v2: 다국어 지원, 384차원
                - sentence-transformers/all-MiniLM-L6-v2: 영어 특화, 384차원
        """
        print(f"임베딩 모델 로딩 중: {model_name}")
        self.encoder = SentenceTransformer(model_name)
        self.dimension = self.encoder.get_sentence_embedding_dimension()
        self.index = None
        self.chunks = []

        print(f"임베딩 모델 로딩 완료 (차원: {self.dimension})")

    def add_texts(self, texts: List[str]):
        """텍스트를 벡터로 변환하여 저장"""
        print(f"{len(texts)}개 청크 임베딩 중...")

        # 임베딩 생성
        embeddings = self.encoder.encode(
            texts,
            show_progress_bar=True,
            convert_to_numpy=True
        )

        # FAISS 인덱스 생성
        self.index = faiss.IndexFlatL2(self.dimension)
        self.index.add(embeddings.astype('float32'))
        self.chunks = texts

        print(f"벡터 저장소 생성 완료: {len(texts)}개 벡터")

    def search(self, query: str, k: int = 3) -> List[tuple]:
        """
        유사한 문서 검색

        Args:
            query: 검색 질문
            k: 반환할 문서 개수

        Returns:
            [(텍스트, 유사도 점수), ...]
        """
        if self.index is None:
            print("❌ 벡터 저장소가 비어있습니다")
            return []

        # 질문 임베딩
        query_embedding = self.encoder.encode([query])

        # 유사도 검색
        distances, indices = self.index.search(
            query_embedding.astype('float32'),
            k
        )

        # 결과 구성
        results = []
        for idx, distance in zip(indices[0], distances[0]):
            if idx < len(self.chunks):
                results.append((self.chunks[idx], float(distance)))

        return results

    def save(self, path: str = "vector_store.index"):
        """벡터 저장소를 파일로 저장"""
        if self.index is None:
            print("저장할 벡터가 없습니다")
            return

        faiss.write_index(self.index, path)

        # 청크도 함께 저장
        import pickle
        with open(f"{path}.chunks", "wb") as f:
            pickle.dump(self.chunks, f)

        print(f"벡터 저장소 저장: {path}")

    def load(self, path: str = "vector_store.index"):
        """저장된 벡터 저장소 로딩"""
        try:
            self.index = faiss.read_index(path)

            import pickle
            with open(f"{path}.chunks", "rb") as f:
                self.chunks = pickle.load(f)

            print(f"벡터 저장소 로딩: {len(self.chunks)}개 벡터")
        except Exception as e:
            print(f"로딩 실패: {e}")

# 사용 예시
vector_store = VectorStore()
vector_store.add_texts(chunks)

# 검색 테스트
#query = "RAG가 무엇인가요?"
query = '교육과 체험학습을 병행한 어린이 안전체험관 운영의 장점은 무엇인가요?'
results = vector_store.search(query, k=2)

print(f"\n질문: {query}")
for i, (text, score) in enumerate(results, 1):
    print(f"\n[검색 결과 {i}] (유사도: {score:.4f})")
    print(text[:100])

임베딩 모델 로딩 중: paraphrase-multilingual-MiniLM-L12-v2


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

임베딩 모델 로딩 완료 (차원: 384)
34개 청크 임베딩 중...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

벡터 저장소 생성 완료: 34개 벡터

질문: 교육과 체험학습을 병행한 어린이 안전체험관 운영의 장점은 무엇인가요?

[검색 결과 1] (유사도: 13.1666)
다.
   ④ 필요시 증거물로 녹화물을 제공한다 .
3. 다음 중 지방자치단체에서 구축한 CCTV 관제센터의 운영과 
거리가 먼 것은?
   ① 자율방범대 , 녹색어머니연합회 , 

[검색 결과 2] (유사도: 14.6911)
손쉽게 열람 또는 제공 등의 주민편익 도모
   ④ 분산,운영 중인 CCTV 시스템의 공간적 ·기능적 통합 및 사
건·사고 발생 시 신속한 대응
4. 법적 근거를 바탕으로 영상정보


In [17]:
from huggingface_hub import login, notebook_login
from dotenv import load_dotenv
import os
load_dotenv()
hf_token = os.getenv('HF_TOKEN')
if hf_token:
    login(token=hf_token)
else:
    notebook_login()
logger.info("huggingface_hub logged in.")

2025-11-03 16:03:46 [I] huggingface_hub logged in.


In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

class LocalLLM:
    """로컬 LLM 래퍼 클래스"""

    def __init__(
        self,
        model_name: str = "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct",
        device: str = "cpu",
        max_length: int = 512
    ):
        """
        Args:
            model_name: HuggingFace 모델 이름
            device: 'cpu' 또는 'cuda'
            max_length: 최대 생성 토큰 수
        """
        print(f"로컬 LLM 로딩 중: {model_name}")
        print(f"   디바이스: {device}")

        # 토크나이저 로딩
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        # 모델 로딩 (양자화 옵션 가능)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
            device_map=device,
            trust_remote_code=True
        )

        # 파이프라인 생성
        self.pipe = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            max_length=max_length,
            temperature=0.3,
            do_sample=True,
            top_p=0.9
        )

        print(f"로컬 LLM 로딩 완료")

    def generate(self, prompt: str) -> str:
        """
        텍스트 생성

        Args:
            prompt: 입력 프롬프트

        Returns:
            생성된 텍스트
        """
        outputs = self.pipe(
            prompt,
            pad_token_id=self.tokenizer.eos_token_id,
            eos_token_id=self.tokenizer.eos_token_id
        )

        # 생성된 텍스트 추출
        generated_text = outputs[0]['generated_text']

        # 프롬프트 부분 제거
        if generated_text.startswith(prompt):
            generated_text = generated_text[len(prompt):].strip()

        return generated_text

# 사용 예시
llm = LocalLLM(device="cpu")

# 간단한 테스트
test_prompt = "RAG 시스템이란 무엇인가요?"
#test_prompt = '교육과 체험학습을 병행한 어린이 안전체험관 운영의 장점은 무엇인가요?'
response = llm.generate(test_prompt)
print(f"질문: {test_prompt}")
print(f"답변: {response}")

로컬 LLM 로딩 중: LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct
   디바이스: cpu


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/4.65G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

Device set to use cpu


로컬 LLM 로딩 완료
질문: RAG 시스템이란 무엇인가요?
답변: RAG 시스템은 인공지능이 사람처럼 질문에 답변하거나 정보를 요약할 수 있도록 설계된 기술입니다. 이 시스템은 자연어 처리(NLP)와 대화형 AI의 핵심 요소를 결합하여, 복잡한 질의응답 시스템이나 챗봇을 구축하는 데 활용됩니다. RAG의 주요 특징은 다음과 같습니다:

1. **질문 이해**: 사용자의 질문을 정확하게 이해하고 분석합니다.
2. **정보 검색**: 다양한 출처에서 관련 정보를 검색하고 수집합니다.
3. **결합 및 요약**: 검색된 정보를 통합하고 사용자에게 의미 있는 답변을 제공합니다.
4. **신뢰성**: 정보의 정확성과 신뢰성을 보장하기 위해 출처 검증을 포함합니다.

RAG 시스템은 의료 진단, 고객 서비스, 교육 등 다양한 분야에서 활용되며, 특히 대량의 데이터를 효과적으로 관리하고 활용할 수 있는 능력이 중요합니다.


In [22]:
!pip install -q bitsandbytes accelerate
!pip install -q -U bitsandbytes

In [24]:
# from transformers import BitsAndBytesConfig

# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,  # 추가: 메모리 최적화
#     bnb_4bit_quant_type="nf4"  # 추가: 양자화 타입 명시
# )

# model_name: str = "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct"
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=quantization_config,
#     device_map="auto",
#     trust_remote_code=True,
#     low_cpu_mem_usage=True  # 추가: CPU 메모리 최적화
# )

In [25]:
class PDFRAGSystem:
    """PDF RAG 시스템 통합 클래스"""

    def __init__(
        self,
        embedding_model: str = "paraphrase-multilingual-MiniLM-L12-v2",
        llm_model: str = "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct",
        device: str = "cpu"
    ):
        """RAG 시스템 초기화"""
        print("=" * 60)
        print("PDF RAG 시스템 초기화 중...")
        print("=" * 60)

        self.vector_store = VectorStore(embedding_model)
        self.llm = LocalLLM(llm_model, device)

        print("=" * 60)
        print("✅ RAG 시스템 준비 완료!")
        print("=" * 60)

    def load_pdf(self, pdf_path: str, chunk_size: int = 500):
        """PDF 로딩 및 인덱싱"""
        print(f"\n📄 PDF 처리 중: {pdf_path}")

        # PDF 텍스트 추출
        text = load_pdf(pdf_path)
        if not text:
            return False

        # 텍스트 분할
        chunks = split_text(text, chunk_size=chunk_size)

        # 벡터 저장소 생성
        self.vector_store.add_texts(chunks)

        return True

    def query(self, question: str, k: int = 3) -> dict:
        """
        질문에 답변

        Args:
            question: 사용자 질문
            k: 검색할 문서 개수

        Returns:
            {
                'answer': 답변,
                'sources': 참고 문서 리스트
            }
        """
        print(f"\n❓ 질문: {question}")

        # 1. 유사 문서 검색
        search_results = self.vector_store.search(question, k=k)

        if not search_results:
            return {
                'answer': "관련 문서를 찾을 수 없습니다.",
                'sources': []
            }

        # 2. 컨텍스트 구성
        context = "\n\n".join([text for text, _ in search_results])

        # 3. 프롬프트 생성
        prompt = f"""다음 문맥을 바탕으로 질문에 답변해주세요. 문맥에 없는 내용은 답변하지 마세요.

문맥:
{context}

질문: {question}

답변:"""

        # 4. LLM 생성
        print("🤔 답변 생성 중...")
        answer = self.llm.generate(prompt)

        return {
            'answer': answer,
            'sources': [text for text, _ in search_results]
        }

    def chat(self):
        """대화형 인터페이스"""
        print("\n" + "=" * 60)
        print("PDF RAG 챗봇 시작")
        print("종료하려면 'quit' 또는 'exit'를 입력하세요")
        print("=" * 60)

        while True:
            try:
                question = input("\n질문: ").strip()

                if question.lower() in ['quit', 'exit', '종료', 'q']:
                    print("👋 챗봇을 종료합니다.")
                    break

                if not question:
                    continue

                result = self.query(question)

                print(f"\n💬 답변:\n{result['answer']}")

                print(f"\n📚 참고 문서:")
                for i, source in enumerate(result['sources'], 1):
                    print(f"  [{i}] {source[:80]}...")

            except KeyboardInterrupt:
                print("\n\n👋 챗봇을 종료합니다.")
                break
            except Exception as e:
                print(f"\n❌ 오류 발생: {e}")

In [27]:
"""
PDF RAG 시스템 MVP (멀티 파일 + 메타데이터)
"""

import PyPDF2
import torch
import faiss
import numpy as np
from typing import List, Dict, Tuple
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter

# =====================================================
# 1. PDF 로딩 (메타데이터 포함)
# =====================================================

def load_pdf_with_metadata(pdf_path: str) -> List[Dict]:
    """
    PDF에서 텍스트 추출 (페이지별 메타데이터 포함)

    Returns:
        [{'text': str, 'file': str, 'page': int}, ...]
    """
    documents = []
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            filename = pdf_path.split('/')[-1]

            for page_num, page in enumerate(pdf_reader.pages, start=1):
                text = page.extract_text()
                if text.strip():
                    documents.append({
                        'text': text,
                        'file': filename,
                        'page': page_num
                    })

        print(f"✅ PDF 로딩: {filename} - {len(documents)}페이지")
        return documents

    except Exception as e:
        print(f"❌ PDF 로딩 실패: {e}")
        return []

# =====================================================
# 2. 텍스트 분할 (메타데이터 유지)
# =====================================================

def split_documents(documents: List[Dict], chunk_size: int = 500) -> List[Dict]:
    """
    문서를 청크로 분할하되 메타데이터 유지

    Returns:
        [{'text': str, 'file': str, 'page': int, 'chunk_id': int}, ...]
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=50,
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    all_chunks = []

    for doc in documents:
        text_chunks = splitter.split_text(doc['text'])

        for chunk_id, chunk_text in enumerate(text_chunks):
            all_chunks.append({
                'text': chunk_text,
                'file': doc['file'],
                'page': doc['page'],
                'chunk_id': chunk_id
            })

    print(f"✅ 텍스트 분할: {len(all_chunks)}개 청크")
    return all_chunks

# =====================================================
# 3. 벡터 저장소 (메타데이터 포함)
# =====================================================

class VectorStore:
    """FAISS 벡터 저장소 with 메타데이터"""

    def __init__(self):
        print("🔄 임베딩 모델 로딩...")
        self.encoder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
        self.dimension = self.encoder.get_sentence_embedding_dimension()
        self.index = None
        self.documents = []  # 메타데이터 포함 문서 저장
        print(f"✅ 임베딩 모델 준비 완료 ({self.dimension}차원)")

    def add_documents(self, documents: List[Dict]):
        """문서 벡터화 및 저장 (메타데이터 포함)"""
        print("🔄 벡터 생성 중...")

        texts = [doc['text'] for doc in documents]
        embeddings = self.encoder.encode(texts, show_progress_bar=True)

        self.index = faiss.IndexFlatL2(self.dimension)
        self.index.add(embeddings.astype('float32'))
        self.documents = documents

        print(f"✅ 벡터 저장 완료: {len(documents)}개")

    def search(self, query: str, k: int = 3) -> List[Dict]:
        """
        유사 문서 검색 (메타데이터 포함 반환)

        Returns:
            [{'text': str, 'file': str, 'page': int, 'score': float}, ...]
        """
        query_vec = self.encoder.encode([query])
        distances, indices = self.index.search(query_vec.astype('float32'), k)

        results = []
        for idx, distance in zip(indices[0], distances[0]):
            if idx < len(self.documents):
                doc = self.documents[idx].copy()
                doc['score'] = float(distance)
                results.append(doc)

        return results

# =====================================================
# 4. 로컬 LLM (변경 없음)
# =====================================================

class LocalLLM:
    """ExaOne 로컬 LLM"""
    global _device

    def __init__(self, device: str = _device):
        print("🔄 로컬 LLM 로딩...")
        model_name = "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct"

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float32,
            device_map=device,
            trust_remote_code=True
        )

        self.pipe = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            max_length=512,
            temperature=0.3
        )
        print("✅ LLM 준비 완료")

    def generate(self, prompt: str) -> str:
        """텍스트 생성"""
        output = self.pipe(prompt, pad_token_id=self.tokenizer.eos_token_id)
        generated = output[0]['generated_text']

        if generated.startswith(prompt):
            generated = generated[len(prompt):].strip()

        return generated

# =====================================================
# 5. RAG 시스템 (멀티 파일 지원)
# =====================================================

class PDFRAG:
    """PDF RAG 시스템 (멀티 파일 + 메타데이터)"""

    def __init__(self):
        print("\n" + "=" * 60)
        print("PDF RAG 시스템 초기화")
        print("=" * 60)

        self.vector_store = VectorStore()
        self.llm = LocalLLM()

        print("=" * 60)
        print("✅ 시스템 준비 완료!")
        print("=" * 60 + "\n")

    def load_pdfs(self, pdf_paths: List[str]):
        """여러 PDF 로딩 및 인덱싱"""
        all_documents = []

        for pdf_path in pdf_paths:
            docs = load_pdf_with_metadata(pdf_path)
            all_documents.extend(docs)

        if not all_documents:
            return False

        # 청크로 분할
        chunks = split_documents(all_documents)

        # 벡터 저장소에 추가
        self.vector_store.add_documents(chunks)
        return True

    def query(self, question: str, k: int = 3) -> Dict:
        """질의응답 (출처 정보 포함)"""
        # 검색
        search_results = self.vector_store.search(question, k=k)

        if not search_results:
            return {
                'answer': "관련 문서를 찾을 수 없습니다.",
                'sources': []
            }

        # 컨텍스트 구성
        context = "\n\n".join([doc['text'] for doc in search_results])

        # 프롬프트
        prompt = f"""다음 문맥을 참고하여 질문에 답변하세요.

문맥:
{context}

질문: {question}

답변:"""

        # 생성
        answer = self.llm.generate(prompt)

        # 출처 정보 포맷팅
        sources = []
        for doc in search_results:
            sources.append({
                'file': doc['file'],
                'page': doc['page'],
                'text': doc['text'][:100] + '...',
                'score': doc['score']
            })

        return {
            'answer': answer,
            'sources': sources
        }

# =====================================================
# 6. 메인 실행
# =====================================================

def main():
    """메인 함수"""

    # RAG 시스템 초기화
    rag = PDFRAG()

    # 여러 PDF 파일 로딩
    pdf_files = [
        "sample_korean.pdf",
        # "data.pdf",  # 추가 파일
        # "another.pdf"
    ]

    if not rag.load_pdfs(pdf_files):
        print("❌ PDF 파일을 로딩할 수 없습니다.")
        return

    # 테스트 질문
    questions = [
        "RAG가 무엇인가요?",
        "ExaOne 모델의 장점은 무엇인가요?",
        "HPGP는 무엇인가요?",
    ]

    print("\n" + "=" * 60)
    print("질의응답 테스트")
    print("=" * 60)

    for i, question in enumerate(questions, 1):
        print(f"\n{'='*60}")
        print(f"질문 {i}: {question}")
        print('='*60)

        result = rag.query(question, k=2)

        print(f"\n💬 답변:\n{result['answer']}")

        print(f"\n📚 참고 문서:")
        for j, source in enumerate(result['sources'], 1):
            print(f"  [{j}] 파일: {source['file']} | 페이지: {source['page']} | 유사도: {source['score']:.4f}")
            print(f"      내용: {source['text']}")

        print()

main()


PDF RAG 시스템 초기화
🔄 임베딩 모델 로딩...
✅ 임베딩 모델 준비 완료 (384차원)
🔄 로컬 LLM 로딩...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda


✅ LLM 준비 완료
✅ 시스템 준비 완료!

✅ PDF 로딩: sample_korean.pdf - 1페이지
✅ 텍스트 분할: 1개 청크
🔄 벡터 생성 중...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ 벡터 저장 완료: 1개

질의응답 테스트

질문 1: RAG가 무엇인가요?

💬 답변:
RAG는 검색 증강 생성 기술로, 대규모 언어 모델의 한계를 극복하기 위해 외부 지식 베이스를 활용하여 생성된 텍스트의 정확성과 최신성을 향상시키는 기술입니다. 주요 특징은 다음과 같습니다:
1. **외부 지식 베이스 활용**: RAG는 사전에 구축된 지식 베이스와 검색 기능을 통해 더 정확하고 신뢰할 수 있는 정보를 제공합니다.
2. **환각(Hallucination) 감소**: 대규모 언어 모델의 한계인 환각 현상을 줄여 실제 정보에 더 가깝게 텍스트를 생성합니다.
3. **최신 정보 제공 가능**: 실시간으로 업데이트되는 지식 베이스를 통해 최신 정보를 반영한 답변을 제공합니다.

ExaOne은 이러한 RAG 기술을 기반으로 개발된 한국어 특화 언어 모델로, 개인 PC에서도 경량화된 형태로 실행 가능합니다.

📚 참고 문서:
  [1] 파일: sample_korean.pdf | 페이지: 1 | 유사도: 12.1860
      내용: 인공지능과 RAG 시스템
RAG(Retrieval-Augmented Generation)는 검색 증강 생성 기술입니다.
이 기술은 대규모 언어 모델의 한계를 극복하기 위해 개발되었...
  [2] 파일: sample_korean.pdf | 페이지: 1 | 유사도: 340282346638528859811704183484516925440.0000
      내용: 인공지능과 RAG 시스템
RAG(Retrieval-Augmented Generation)는 검색 증강 생성 기술입니다.
이 기술은 대규모 언어 모델의 한계를 극복하기 위해 개발되었...


질문 2: ExaOne 모델의 장점은 무엇인가요?

💬 답변:
ExaOne 모델의 장점은 다음과 같습니다:

1. **경량화된 크기**: ExaOne은 대규모 모델에 비해 상대적으로 작은 크기로 설계되어 개인 PC에서도 효율적으로 실행 가능합니다. 이는 컴퓨팅 자원의 제약이 있는 환경에서도 모

In [30]:
# 셀 2 수정
!pip install -q torch transformers sentence-transformers pypdf2 faiss-cpu langchain tqdm
!pip install -q langchain-core langchain-community  # 추가
!pip install -q pypdf  # 추가 (PyPDF2 대신)
print("Installed required packages.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 13.7 MB/s eta 0:00:00
Installed required packages.


In [31]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

class PDFRAGwithLangChain:
    """셀 18 기반 LangChain LCEL RAG 시스템"""

    def __init__(self):
        global _device
        print("=" * 60)
        print("LangChain LCEL RAG 시스템 초기화")
        print("=" * 60)

        # 임베딩 모델
        self.embeddings = HuggingFaceEmbeddings(
            model_name='paraphrase-multilingual-MiniLM-L12-v2'
        )

        # LLM 파이프라인
        from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
        model_name = "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct"
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float32,
            device_map=_device,
            trust_remote_code=True
        )

        pipe = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            max_length=512,
            temperature=0.3
        )

        self.llm = HuggingFacePipeline(pipeline=pipe)

        # 프롬프트 템플릿
        self.prompt = PromptTemplate(
            template="""다음 문맥을 참고하여 질문에 답변하세요.

문맥:
{context}

질문: {question}

답변:""",
            input_variables=["context", "question"]
        )

        self.vectorstore = None
        self.rag_chain = None

        print("✅ 시스템 준비 완료!\n")

    def load_pdfs(self, pdf_paths: List[str]):
        """PDF 로딩 (메타데이터 포함 Document 생성)"""
        all_docs = []

        for pdf_path in pdf_paths:
            # 기존 load_pdf_with_metadata 함수 재사용
            page_docs = load_pdf_with_metadata(pdf_path)

            for doc_dict in page_docs:
                # Document 객체로 변환
                doc = Document(
                    page_content=doc_dict['text'],
                    metadata={
                        'file': doc_dict['file'],
                        'page': doc_dict['page']
                    }
                )
                all_docs.append(doc)

        if not all_docs:
            return False

        # 텍스트 분할 (LangChain 방식)
        from langchain_text_splitters import RecursiveCharacterTextSplitter
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=50
        )
        splits = splitter.split_documents(all_docs)

        print(f"✅ {len(splits)}개 청크 생성")

        # 벡터 저장소 생성
        self.vectorstore = FAISS.from_documents(splits, self.embeddings)

        # LCEL 체인 구성
        self._create_chain()
        return True

    def _create_chain(self):
        """LCEL 체인 생성"""
        retriever = self.vectorstore.as_retriever(search_kwargs={"k": 3})

        def format_docs(docs):
            return "\n\n".join(doc.page_content for doc in docs)

        self.rag_chain = (
            {
                "context": retriever | format_docs,
                "question": RunnablePassthrough()
            }
            | self.prompt
            | self.llm
            | StrOutputParser()
        )

        print("✅ LCEL 체인 생성 완료")

    def query(self, question: str, k: int = 3) -> Dict:
        """질의응답 (출처 정보 포함)"""
        if self.rag_chain is None:
            return {'answer': "시스템 미초기화", 'sources': []}

        # 검색
        retriever = self.vectorstore.as_retriever(search_kwargs={"k": k})
        source_docs = retriever.invoke(question)

        # 답변 생성
        answer = self.rag_chain.invoke(question)

        # 출처 포맷팅
        sources = []
        for doc in source_docs:
            sources.append({
                'file': doc.metadata.get('file', 'Unknown'),
                'page': doc.metadata.get('page', 0),
                'text': doc.page_content[:100] + '...',
                'score': 0.0  # FAISS는 거리 반환, 유사도는 별도 계산 필요
            })

        return {'answer': answer, 'sources': sources}

# 실행
def main_lcel():
    rag = PDFRAGwithLangChain()

    pdf_files = ["sample_korean.pdf"]

    if not rag.load_pdfs(pdf_files):
        print("❌ PDF 로딩 실패")
        return

    questions = [
        "RAG가 무엇인가요?",
        "ExaOne 모델의 장점은 무엇인가요?",
        "HPGP 무엇인가요?",
    ]

    for question in questions:
        print(f"\n질문: {question}")
        result = rag.query(question, k=2)
        print(f"답변: {result['answer']}\n")
        print("📚 참고 문서:")
        for i, src in enumerate(result['sources'], 1):
            print(f"  [{i}] {src['file']} p.{src['page']}")

main_lcel()

LangChain LCEL RAG 시스템 초기화


/tmp/ipython-input-1333280680.py:19: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings = HuggingFaceEmbeddings(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda
/tmp/ipython-input-1333280680.py:42: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  self.llm = HuggingFacePipeline(pipeline=pipe)


✅ 시스템 준비 완료!

✅ PDF 로딩: sample_korean.pdf - 1페이지
✅ 1개 청크 생성
✅ LCEL 체인 생성 완료

질문: RAG가 무엇인가요?
답변: 다음 문맥을 참고하여 질문에 답변하세요.

문맥:
인공지능과 RAG 시스템
RAG(Retrieval-Augmented Generation)는 검색 증강 생성 기술입니다.
이 기술은 대규모 언어 모델의 한계를 극복하기 위해 개발되었습니다.
주요 특징:
1. 외부 지식 베이스 활용
2. 환각(Hallucination) 감소
3. 최신 정보 제공 가능
ExaOne은 LG AI연구원이 개발한 한국어 언어 모델입니다.
경량화된 크기로 개인 PC에서도 실행 가능합니다.

질문: RAG가 무엇인가요?

답변:
RAG는 검색 증강 생성(Retrieval-Augmented Generation) 기술로, 대규모 언어 모델의 한계를 극복하기 위해 외부 지식 베이스를 활용하여 생성 과정을 보완하는 기술입니다. 주요 특징은 다음과 같습니다:
1. **외부 지식 베이스 활용**: 기존의 정보나 데이터베이스를 통해 더 정확하고 신뢰할 수 있는 정보를 제공합니다.
2. **환각(Hallucination) 감소**: 모델이 사실과 다른 정보를 생성하는 환각 현상을 줄여줍니다.
3. **최신 정보 제공**: 실시간으로 업데이트되는 지식 베이스를 통해 최신 정보를 반영할 수 있습니다.

RAG 기술은 특히 ExaOne과 같은 한국어 언어 모델에서도 적용되어, 더 나은 정보 검색 및 생성 능력을 제공합니다.

📚 참고 문서:
  [1] sample_korean.pdf p.1

질문: ExaOne 모델의 장점은 무엇인가요?
답변: 다음 문맥을 참고하여 질문에 답변하세요.

문맥:
인공지능과 RAG 시스템
RAG(Retrieval-Augmented Generation)는 검색 증강 생성 기술입니다.
이 기술은 대규모 언어 모델의 한계를 극복하기 위해 개발되었습니다.
주요 특징:
1. 외부 지식 베이스 활용
2. 환각(Halluc

In [32]:
!pip install -q langchain-core langchain-community
!pip install pypdf

In [33]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline

# 수정: langchain_core로 변경
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
from typing import List, Dict
from pathlib import Path

# =====================================================
# 1. LangChain LCEL 기반 RAG 시스템
# =====================================================

class LangChainPDFRAG:
    """LangChain LCEL 기반 PDF RAG 시스템"""
    global _device

    def __init__(
        self,
        embedding_model: str = "paraphrase-multilingual-MiniLM-L12-v2",
        llm_model: str = "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct",
        device: str = _device,
        chunk_size: int = 500,
        chunk_overlap: int = 50
    ):
        print("\n" + "=" * 60)
        print("LangChain LCEL RAG 시스템 초기화")
        print("=" * 60)

        # 임베딩 모델 초기화
        print("🔄 임베딩 모델 로딩...")
        self.embeddings = HuggingFaceEmbeddings(
            model_name=embedding_model,
            model_kwargs={'device': device}
        )

        # 텍스트 분할기 초기화
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ". ", " ", ""]
        )

        # LLM 초기화
        print("🔄 로컬 LLM 로딩...")
        tokenizer = AutoTokenizer.from_pretrained(llm_model)
        model = AutoModelForCausalLM.from_pretrained(
            llm_model,
            torch_dtype=torch.float32,
            device_map=device,
            trust_remote_code=True
        )

        pipe = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            max_length=512,
            temperature=0.3,
            do_sample=True,
            top_p=0.9
        )

        self.llm = HuggingFacePipeline(pipeline=pipe)

        # 프롬프트 템플릿
        self.prompt_template = PromptTemplate(
            template="""다음 문맥을 참고하여 질문에 답변하세요.
문맥에 없는 내용은 답변하지 마세요.

문맥:
{context}

질문: {question}

답변:""",
            input_variables=["context", "question"]
        )

        # 출력 파서
        self.output_parser = StrOutputParser()

        self.vectorstore = None
        self.retriever = None
        self.rag_chain = None

        print("=" * 60)
        print("✅ 시스템 준비 완료!")
        print("=" * 60 + "\n")

    def load_pdfs(self, pdf_paths: List[str]) -> bool:
        """여러 PDF 파일 로딩 및 벡터 저장소 생성"""
        all_documents = []

        for pdf_path in pdf_paths:
            try:
                print(f"📄 로딩 중: {pdf_path}")
                loader = PyPDFLoader(pdf_path)
                documents = loader.load()
                all_documents.extend(documents)
                print(f"✅ {len(documents)}페이지 로딩 완료")
            except Exception as e:
                print(f"❌ 로딩 실패 ({pdf_path}): {e}")

        if not all_documents:
            print("❌ 로딩된 문서가 없습니다.")
            return False

        # 텍스트 분할
        print(f"\n🔄 텍스트 분할 중...")
        splits = self.text_splitter.split_documents(all_documents)
        print(f"✅ {len(splits)}개 청크 생성")

        # 벡터 저장소 생성
        print(f"\n🔄 벡터 저장소 생성 중...")
        self.vectorstore = FAISS.from_documents(
            documents=splits,
            embedding=self.embeddings
        )
        print(f"✅ 벡터 저장소 생성 완료")

        # Retriever 설정
        self.retriever = self.vectorstore.as_retriever(
            search_kwargs={"k": 3}
        )

        # LCEL 체인 구성
        self._create_rag_chain()

        return True

    def _create_rag_chain(self):
        """
        LCEL 방식으로 RAG 체인 구성
        chain = prompt | llm | output_parser
        """
        def format_docs(docs):
            """검색된 문서들을 문자열로 포맷팅"""
            return "\n\n".join(doc.page_content for doc in docs)

        # LCEL 체인: Retriever → Prompt → LLM → Output Parser
        self.rag_chain = (
            {
                "context": self.retriever | format_docs,
                "question": RunnablePassthrough()
            }
            | self.prompt_template
            | self.llm
            | self.output_parser
        )

        print("✅ LCEL RAG 체인 생성 완료")

    def query(self, question: str, k: int = 3) -> Dict:
        """
        질의응답 (출처 정보 포함)

        Args:
            question: 사용자 질문
            k: 검색할 문서 개수

        Returns:
            {'answer': str, 'sources': List[Dict]}
        """
        if self.rag_chain is None:
            return {
                'answer': "벡터 저장소가 초기화되지 않았습니다.",
                'sources': []
            }

        # 검색 개수 업데이트
        self.retriever.search_kwargs = {"k": k}

        # 수정: get_relevant_documents → invoke
        source_docs = self.retriever.invoke(question)

        # LCEL 체인 실행
        answer = self.rag_chain.invoke(question)

        # 출처 정보 포맷팅
        sources = []
        for doc in source_docs:
            sources.append({
                'file': doc.metadata.get('source', 'Unknown'),
                'page': doc.metadata.get('page', 0) + 1,
                'text': doc.page_content[:100] + '...',
                'score': 0.0
            })

        return {
            'answer': answer,
            'sources': sources
        }

    def query_with_chain(self, question: str) -> str:
        """
        LCEL 체인만 사용한 간단한 질의
        (출처 정보 없음)
        """
        if self.rag_chain is None:
            return "벡터 저장소가 초기화되지 않았습니다."

        return self.rag_chain.invoke(question)

    def chat(self):
        """대화형 인터페이스"""
        print("\n" + "=" * 60)
        print("PDF RAG 챗봇 시작 (LCEL 방식)")
        print("종료하려면 'quit' 또는 'exit'를 입력하세요")
        print("=" * 60)

        while True:
            try:
                question = input("\n❓ 질문: ").strip()

                if question.lower() in ['quit', 'exit', '종료', 'q']:
                    print("👋 챗봇을 종료합니다.")
                    break

                if not question:
                    continue

                result = self.query(question)

                print(f"\n💬 답변:\n{result['answer']}")

                print(f"\n📚 참고 문서:")
                for i, source in enumerate(result['sources'], 1):
                    print(f"  [{i}] 파일: {source['file']} | 페이지: {source['page']}")
                    print(f"      내용: {source['text']}")

            except KeyboardInterrupt:
                print("\n\n👋 챗봇을 종료합니다.")
                break
            except Exception as e:
                print(f"\n❌ 오류 발생: {e}")

# =====================================================
# 2. 메인 실행
# =====================================================

def main():
    """메인 함수"""

    # RAG 시스템 초기화
    rag = LangChainPDFRAG(
        chunk_size=500,
        chunk_overlap=50,
        device="cpu"
    )

    # PDF 파일 로딩
    data_pdf_path = str(Path(drive_root()) / 'data' / 'data.pdf')
    pdf_files = [
        # data_pdf_path,
        "sample_korean.pdf",  # 추가 파일
    ]

    if not rag.load_pdfs(pdf_files):
        print("❌ PDF 파일을 로딩할 수 없습니다.")
        return

    # 테스트 질문
    questions = [
        "교육과 체험학습을 병행한 어린이 안전체험관 운영의 장점은 무엇인가요?",
        "HPGP는 무엇인가요?",
    ]

    print("\n" + "=" * 60)
    print("질의응답 테스트 (LCEL 방식)")
    print("=" * 60)

    for i, question in enumerate(questions, 1):
        print(f"\n{'='*60}")
        print(f"질문 {i}: {question}")
        print('='*60)

        result = rag.query(question, k=2)

        print(f"\n💬 답변:\n{result['answer']}")

        print(f"\n📚 참고 문서:")
        for j, source in enumerate(result['sources'], 1):
            print(f"  [{j}] 파일: {source['file']} | 페이지: {source['page']}")
            print(f"      내용: {source['text']}")

        print()

    # 대화형 모드 (선택사항)
    # rag.chat()

main()


LangChain LCEL RAG 시스템 초기화
🔄 임베딩 모델 로딩...
🔄 로컬 LLM 로딩...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cpu


✅ 시스템 준비 완료!

📄 로딩 중: sample_korean.pdf
✅ 1페이지 로딩 완료

🔄 텍스트 분할 중...
✅ 1개 청크 생성

🔄 벡터 저장소 생성 중...
✅ 벡터 저장소 생성 완료
✅ LCEL RAG 체인 생성 완료

질의응답 테스트 (LCEL 방식)

질문 1: 교육과 체험학습을 병행한 어린이 안전체험관 운영의 장점은 무엇인가요?

💬 답변:
다음 문맥을 참고하여 질문에 답변하세요. 
문맥에 없는 내용은 답변하지 마세요.

문맥:
인공지능과 RAG 시스템
RAG(Retrieval-Augmented Generation)는 검색 증강 생성 기술입니다.
이 기술은 대규모 언어 모델의 한계를 극복하기 위해 개발되었습니다.
주요 특징:
1. 외부 지식 베이스 활용
2. 환각(Hallucination) 감소
3. 최신 정보 제공 가능
ExaOne은 LG AI연구원이 개발한 한국어 언어 모델입니다.
경량화된 크기로 개인 PC에서도 실행 가능합니다.

질문: 교육과 체험학습을 병행한 어린이 안전체험관 운영의 장점은 무엇인가요?

답변:
교육과 체험학습을 병행한 어린이 안전체험관 운영의 장점은 다음과 같습니다:

1. **실용적 지식 습득**: 실제 상황을 체험함으로써 이론적인 지식을 실제 적용 능력으로 연결시킵니다. 예를 들어, 화재 대피 훈련을 통해 안전 수칙을 실제로 이해하고 익힐 수 있습니다.
2. **참여와 동기 부여**: 체험을 통해 어린이들이 적극적으로 참여하게 되어 학습에 대한 흥미와 동기를 높일 수 있습니다. 이는 수동적인 학습 방식보다 더 효과적인 지식 습득을 가능하게 합니다.
3. **위험 인식 향상**: 다양한 위험 상황을 직접 경험함으로써 어린이들이 위험을 인식하고 대처 방법을 미리 배울 수 있습니다. 이는 안전 의식을 강화하고 실제 위험 상황에서의 대처 능력을 향상시킵니다.
4. **사회적 기술 개발**: 협동과 소통을 통해 다른 어린이들과 함께 문제를 해결하는 과정에서 사회적 기술을 개발할 수 있습니다. 이는 미래 사회에서 필요